In [ ]:
%pip install imageai
%pip install tensorflow keras opencv-python


^C
Note: you may need to restart the kernel to use updated packages.


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Parameters
img_height, img_width = 224, 224
batch_size = 32
epochs = 20


In [5]:
# Data Generator with Validation Split
data_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # Reserve 20% of the data for validation
)

# Training Data
train_data = data_gen.flow_from_directory(
    "D:\\Data\\train",
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'  # Use 80% of the data for training
)

# Validation Data
validation_data = data_gen.flow_from_directory(
    "D:\\Data\\train",
    target_size=(img_height, img_width), 
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'  # Use 20% of the data for validation
)


Found 10987 images belonging to 2 classes.
Found 2746 images belonging to 2 classes.


In [6]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(2, activation='softmax')  # 2 classes: fire and smoke
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [7]:
# Train the Model
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=epochs 
) 


Epoch 1/20


KeyError: 'sequential/conv2d/Conv2D'

In [8]:
# Evaluate on the validation set
val_loss, val_accuracy = model.evaluate(validation_data)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")


12/86 [===>..........................] - ETA: 3:22 - loss: 0.6927 - accuracy: 0.5260

KeyboardInterrupt: 

In [9]:
model.save("smoke_fire_detection_model.h5")
print("Model saved successfully!")


Model saved successfully!


In [18]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

def predict_image(image_path):
    img = load_img(image_path, target_size=(224, 224))  # Replace with actual target size used in training
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    prediction = model.predict(img_array)
    class_names = ['fire', 'smoke']  # Adjust if you changed class names
    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction) * 100

    print(f"Prediction: {predicted_class} with {confidence:.2f}% confidence")
    return predicted_class, confidence

# Example usage
predict_image(r"D:\\Honors\\Smoke-and-Fire-Recognition-main\\Data\\Training Data\\train-smoke\\000011.jpg")
predict_image(r"D:\\Honors\\Smoke-and-Fire-Recognition-main\\Data\\test_small\\test_fire2.jpg")
predict_image(r"D:\\Honors\\Smoke-and-Fire-Recognition-main\\Data\\Training Data\\train_fire\\fire-45.7396875153.png")


1/1 [==============================] - 0s 118ms/step
Prediction: smoke with 50.13% confidence
1/1 [==============================] - 0s 72ms/step
Prediction: smoke with 51.72% confidence
1/1 [==============================] - 0s 73ms/step
Prediction: smoke with 51.03% confidence


('smoke', 51.03379487991333)

In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# Load your trained model
model = load_model("smoke_fire_detection_model.h5")

# Define class names based on your dataset labels
class_names = ['fire', 'smoke']  # Adjust as per your dataset

# Define target image size based on your model input
img_height, img_width = 224, 224  # Replace with the target size used in training

# Start video capture
cap = cv2.VideoCapture(0)  # Use 0 for webcam, or replace with video file path

while cap.isOpened():
    # Capture frame-by-frame
    ret, frame = cap.read()
    if not ret:
        break

    # Preprocess the frame
    img = cv2.resize(frame, (img_height, img_width))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Make prediction
    prediction = model.predict(img_array)
    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction) * 100

    # Display the results on the frame
    label = f"{predicted_class}: {confidence:.2f}%"
    color = (0, 0, 255) if predicted_class == "fire" else (0, 255, 0)  # Red for fire, green for smoke
    cv2.putText(frame, label, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    # Display the resulting frame
    cv2.imshow("Smoke and Fire Detection", frame)

    # Press 'q' to exit the video capture
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the capture and close windows
cap.release()
cv2.destroyAllWindows()


1/1 [==============================] - 0s 97ms/step
